In [13]:
import openpyxl
import glob
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from datetime import datetime
from copy import copy

# Prompt for the client name
client_name = input("Enter the client name: ")

# Define the patterns for the files to be consolidated
file_patterns = ["Step 3*", "Step 4*", "Step 5*", "Step 6*", "Step 7*"]

# Initialize a list to store file paths
file_paths = []

# Iterate over each pattern to find matching files
for pattern in file_patterns:
    file_paths.extend(glob.glob(pattern + "*.xlsx"))

# If no files were found, stop execution
if not file_paths:
    print("No files found matching the patterns.")
    exit()

# Create the output filename
output_filename = f"{client_name} - SDP Reporting Package - {datetime.today().strftime('%Y-%m-%d')}.xlsx"

# Create a new workbook
output_wb = openpyxl.Workbook()

# Remove the default sheet created by openpyxl
if "Sheet" in output_wb.sheetnames:
    output_wb.remove(output_wb["Sheet"])

# Function to copy column widths
def copy_column_widths(input_sheet, output_sheet):
    for col_idx, col_dim in input_sheet.column_dimensions.items():
        # Preserve the original width if set in the input file
        if col_dim.width is not None:
            output_sheet.column_dimensions[col_idx].width = col_dim.width

# Function to apply row-wide formatting (highlighting) even to empty cells
def apply_row_wide_formatting(input_sheet, output_sheet):
    # Loop through each row
    for row in input_sheet.iter_rows():
        row_fill = None
        # Check if any cell in the row has a fill (background color)
        for cell in row:
            if cell.fill and cell.fill.fgColor and cell.fill.fgColor.rgb != '00000000':
                row_fill = cell.fill  # Store the fill if found
                break  # We only need to check one cell to apply the fill to the whole row
        
        # If a fill was found, apply it to the entire row
        if row_fill:
            for col_idx in range(1, input_sheet.max_column + 1):
                output_cell = output_sheet.cell(row=row[0].row, column=col_idx)
                output_cell.fill = copy(row_fill)

# Function to copy conditional formatting
def copy_conditional_formatting(input_sheet, output_sheet):
    # Iterate over all conditional formatting rules in the input sheet
    for cf in input_sheet.conditional_formatting._cf_rules:
        # Each 'cf' is a list of rules, iterate over them
        for rule in input_sheet.conditional_formatting._cf_rules[cf]:
            # Add each rule to the output sheet, with the correct ranges
            output_sheet.conditional_formatting.add(cf, rule)

# Function to rename specific tabs
def rename_tabs(output_wb):
    tab_renames = {
        'Summary': 'Teal iQ Summary',
        'data': 'Diversity Data',
        'Certification Spend': 'Cert Spend',
        'Diversity Report': 'Diversity Trust Scores'
    }
    
    for sheet in output_wb.sheetnames:
        if sheet in tab_renames:
            output_wb[sheet].title = tab_renames[sheet]

# Function to apply tab colors
def apply_tab_colors(output_wb):
    tab_colors = {
        'NAICS List': 'FF0000',  # red
        'NAICS Codes': 'FF0000',  # red
        'NAICS Pivot': 'FF0000',  # red
        'Teal iQ Summary': 'FFFF00',  # yellow
        'Unmatched Data': '800080',  # purple
        'Unmatched Summary': '800080',  # purple
        'Firm vs Establishment': 'FFA500',  # orange
        'Consolidated Rows': 'FFA500',  # orange
        'Consolidated Summary': 'FFA500',  # orange
        'Diversity Data': '0000FF',  # blue
        'Cert Spend': '0000FF',  # blue
        'Diversity Trust Scores': '0000FF',  # blue
    }
    
    for sheet in output_wb.sheetnames:
        if sheet in tab_colors:
            output_wb[sheet].sheet_properties.tabColor = tab_colors[sheet]

# Loop through each found file
for file_path in file_paths:
    # Load the workbook using openpyxl
    input_wb = load_workbook(file_path)

    # Loop through all the sheets in the input workbook
    for sheet_name in input_wb.sheetnames:
        # Access the sheet from the input workbook
        input_sheet = input_wb[sheet_name]

        # Create a new sheet in the output workbook with the same name
        output_sheet = output_wb.create_sheet(title=sheet_name)

        # Copy column widths
        copy_column_widths(input_sheet, output_sheet)

        # Copy all cells and their formatting from input to output
        max_column = input_sheet.max_column
        max_row = input_sheet.max_row
        for row in input_sheet.iter_rows(min_row=1, max_row=max_row, max_col=max_column):
            for cell in row:
                # Copy the cell value
                output_cell = output_sheet.cell(row=cell.row, column=cell.column, value=cell.value)
                
                # Copy cell styles and formatting
                if cell.has_style:
                    output_cell.font = copy(cell.font)
                    output_cell.border = copy(cell.border)
                    output_cell.fill = copy(cell.fill)
                    output_cell.number_format = copy(cell.number_format)
                    output_cell.protection = copy(cell.protection)
                    output_cell.alignment = copy(cell.alignment)

        # Ensure row-wide formatting (like row highlights) is applied
        apply_row_wide_formatting(input_sheet, output_sheet)

        # Copy conditional formatting from input sheet to output sheet
        copy_conditional_formatting(input_sheet, output_sheet)

# Apply tab renaming
rename_tabs(output_wb)

# Apply tab colors
apply_tab_colors(output_wb)

# Save the output workbook to the output filename
output_wb.save(output_filename)

print(f"Consolidation complete. The output file is saved as '{output_filename}'.")


Enter the client name:  CISCO


Consolidation complete. The output file is saved as 'CISCO - SDP Reporting Package - 2024-10-18.xlsx'.
